# ATI Project — Music Genre Classification (FMA-small)
## Phase 1 · Feature extraction (LOCAL / Jupyter)

### 1. Requirements

In [ ]:
import os, warnings, ast
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
warnings.filterwarnings("ignore")
print("librosa", librosa.__version__)

### 2. Paths and parameters

In [ ]:
# Paths and parameters
AUDIO_DIR    = "C:/Users/Francisco Javier/Desktop/Universidad/Integración de tecnologías/Proyecto/files/fma_small/fma_small"
METADATA_DIR = "C:/Users/Francisco Javier/Desktop/Universidad/Integración de tecnologías/Proyecto/files/fma_metadata/fma_metadata"
OUT_CSV = "fma_small_features.csv"

SR       = 22050
DURATION = 30.0
N_MFCC   = 20

### 3. Loading utilities

In [ ]:
def load_tracks(metadata_dir):
    """Load tracks.csv with its multi-level header (adapted from mdeff/fma utils.py)."""
    path = os.path.join(metadata_dir, "tracks.csv")
    tracks = pd.read_csv(path, index_col=0, header=[0, 1])
    tracks[('track', 'genre_top')] = tracks[('track', 'genre_top')].astype('category')
    tracks[('set', 'subset')]      = tracks[('set', 'subset')].astype('category')
    tracks[('set', 'split')]       = tracks[('set', 'split')].astype('category')
    return tracks

def get_audio_path(audio_dir, track_id):
    """Rebuild the mp3 path from a track_id (123 -> audio_dir/000/000123.mp3)."""
    tid = f"{track_id:06d}"
    return os.path.join(audio_dir, tid[:3], tid + ".mp3")

### 4. Select the *small* subset and its official split

In [ ]:
tracks = load_tracks(METADATA_DIR)

small = tracks[tracks[('set', 'subset')] == 'small'].copy()
meta = pd.DataFrame({
    "track_id": small.index,
    "genre":    small[('track', 'genre_top')].values,
    "split":    small[('set', 'split')].values,
})
print("Tracks in FMA-small:", len(meta))
print("\nGenres:\n", meta['genre'].value_counts())
print("\nOfficial split:\n", meta['split'].value_counts())

### 5. Known corrupt tracks

In [ ]:
CORRUPT = {98565, 98567, 98569, 99134, 108925, 133297}

### 6. Feature extraction function

In [ ]:
def _stats(x):
    """Mean and std over time -> fixed-length vector."""
    return np.concatenate([x.mean(axis=1), x.std(axis=1)])

def extract_features(y, sr):
    """Build the 89-value feature vector (timbre, harmony, rhythm) for one track."""
    f = {}
    f['mfcc']     = _stats(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC))
    f['chroma']   = _stats(librosa.feature.chroma_stft(y=y, sr=sr))
    f['contrast'] = _stats(librosa.feature.spectral_contrast(y=y, sr=sr))
    f['centroid'] = _stats(librosa.feature.spectral_centroid(y=y, sr=sr))
    f['bandwidth']= _stats(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    f['rolloff']  = _stats(librosa.feature.spectral_rolloff(y=y, sr=sr))
    f['zcr']      = _stats(librosa.feature.zero_crossing_rate(y))
    f['rms']      = _stats(librosa.feature.rms(y=y))
    tempo, _      = librosa.beat.beat_track(y=y, sr=sr)
    f['tempo']    = np.array([float(np.atleast_1d(tempo)[0])])
    return np.concatenate(list(f.values()))

def feature_names():
    """Column names matching the feature vector layout."""
    names = []
    names += [f"mfcc{i}_mean" for i in range(N_MFCC)]      + [f"mfcc{i}_std" for i in range(N_MFCC)]
    names += [f"chroma{i}_mean" for i in range(12)]        + [f"chroma{i}_std" for i in range(12)]
    names += [f"contrast{i}_mean" for i in range(7)]       + [f"contrast{i}_std" for i in range(7)]
    for base in ["centroid","bandwidth","rolloff","zcr","rms"]:
        names += [f"{base}_mean", f"{base}_std"]
    names += ["tempo"]
    return names

COLS = feature_names()
print("Feature vector size:", len(COLS))

### 7. Extraction loop (incremental saving and resume)

In [ ]:
def already_done(out_csv):
    """Return the set of track_ids already written to the CSV."""
    if os.path.exists(out_csv):
        done = pd.read_csv(out_csv, usecols=["track_id"])["track_id"].tolist()
        return set(done)
    return set()

done = already_done(OUT_CSV)
write_header = not os.path.exists(OUT_CSV)

rows = meta[~meta["track_id"].isin(done) & ~meta["track_id"].isin(CORRUPT)]
print(f"Pending: {len(rows)}  |  Already done: {len(done)}")

with open(OUT_CSV, "a", newline="") as fh:
    if write_header:
        fh.write("track_id," + ",".join(COLS) + ",genre,split\n")
    for _, r in tqdm(rows.iterrows(), total=len(rows)):
        tid = int(r["track_id"])
        path = get_audio_path(AUDIO_DIR, tid)
        y, sr = librosa.load(path, sr=SR, mono=True, duration=DURATION)
        if y.size < sr:
            continue
        vec = extract_features(y, sr)
        line = [str(tid)] + [f"{v:.6g}" for v in vec] + [str(r["genre"]), str(r["split"])]
        fh.write(",".join(line) + "\n"); fh.flush()

print("\nDone.")

### 8. Check the result

In [ ]:
OUT_CSV = "fma_small_features.csv"
df = pd.read_csv(OUT_CSV)
print("Feature dataset shape:", df.shape)
print("\nGenre counts:\n", df["genre"].value_counts())
print("\nSplit counts:\n", df["split"].value_counts())
df.head()

### Next step
Upload `fma_small_features.csv` to Google Colab and open the Phase 2 notebook.